In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

# Move to repo root so all paths work
_p = Path.cwd()
while not (_p / "demos").exists() and _p != _p.parent:
    _p = _p.parent
if (_p / "demos").exists():
    os.chdir(_p)

load_dotenv()
DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Audio

What can you do with audio? Turn it into text, then do text things. Split it by speaker.

## Transcribe and diarize (local)

WhisperX transcribes the audio and identifies who said what. Free, runs locally, your audio never leaves your machine.

**Setup:** Diarization requires a free Hugging Face token (`HF_TOKEN`) and accepting the model licenses at [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) and [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1).


**`audio/whisperx-diarize.py`** — Full WhisperX pipeline: transcribe, align, diarize (who said what)


In [ ]:
from pathlib import Path
import os
from collections import defaultdict
import torch, whisperx
from whisperx.diarize import DiarizationPipeline

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL, LANGUAGE = "large-v3", "en"
HF_TOKEN = os.environ["HF_TOKEN"]
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

# Step 1: Transcribe
model = whisperx.load_model(MODEL, device, compute_type=compute_type)
audio = whisperx.load_audio(str(AUDIO))
result = model.transcribe(audio, batch_size=16)
# Step 2: Align
model_a, metadata = whisperx.load_align_model(language_code=LANGUAGE, device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
# Step 3: Diarize
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
result = whisperx.assign_word_speakers(diarize_model(audio), result)

# Print speaker-labeled segments
current_speaker = None
for seg in result["segments"]:
    speaker = seg.get("speaker", "UNKNOWN")
    if speaker != current_speaker:
        print(f"\n--- {speaker} ---")
        current_speaker = speaker
    start = f"{int(seg['start']//60):02d}:{seg['start']%60:05.2f}"
    end = f"{int(seg['end']//60):02d}:{seg['end']%60:05.2f}"
    print(f"  [{start} - {end}] {seg['text'].strip()}")

# Speaker time summary
speaker_time = defaultdict(float)
for seg in result["segments"]:
    speaker_time[seg.get("speaker", "UNKNOWN")] += seg["end"] - seg["start"]
total = sum(speaker_time.values())
print(f"\n{'Speaker':<12} {'Time':>8} {'%':>6}")
for spk in sorted(speaker_time):
    t = speaker_time[spk]
    print(f"  {spk:<10} {t/60:>6.1f}m {t/total*100:>5.1f}%")


"Speaker 1 said X at 0:42, Speaker 2 said Y at 1:15." Now you have a searchable, speaker-labeled transcript.

## Structured transcription (cloud)

Same audio, but Gemini returns structured data — each utterance as a typed object with speaker, timestamps, and text. Same Pydantic AI pattern as the image notebooks.


**`audio/gemini-diarize.py`** — Structured transcription with speaker labels via Gemini (cloud alternative to WhisperX)


In [ ]:
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
AUDIO = DATA / "rDXubdQdJYs.mp3"
MODEL = "google-gla:gemini-2.5-flash"

class Utterance(BaseModel):
    speaker: str = Field(description="Speaker label (e.g., Speaker 1, Speaker 2)")
    start: str = Field(description="Start timestamp MM:SS")
    end: str = Field(description="End timestamp MM:SS")
    text: str = Field(description="What was said")

agent = Agent(MODEL, output_type=list[Utterance])
result = agent.run_sync([
    "Transcribe this audio with speaker labels and timestamps for each utterance.",
    BinaryContent(data=AUDIO.read_bytes(), media_type="audio/mpeg"),
])
for u in result.output:
    print(f"[{u.start} - {u.end}] {u.speaker}: {u.text}")


Cloud tradeoff: faster, structured output built-in, but your audio goes to Google. Most newsrooms will use both local and cloud depending on the sensitivity of the material.
